# CSE476 Agentic AI and Intelligent Automation
## CA1 Project 1: Study Planner Agent (Topic T24)
**Goal:** Plan study blocks around real deadlines using a custom plan-act loop, memory, and python-based tools.

### Project Requirements Checklist:
- **One Agent with a plan-act loop:** Executes multiple steps autonomously based on tool results.
- **At least two tools:** `add_task(name, due)` and `build_schedule()` implemented as Python functions operating on state.
- **Memory:** Stores tasks and deadlines across turns and reuse them later.
- **Plain Python Agent:** Built from scratch without high-level agent frameworks.

### Step 1: Setup & Installations
We install the Google GenAI SDK (`google-genai`) and setup the API Key. If the API key is not present in the environment, we will prompt you to enter it safely.

In [ ]:
# Install GenAI SDK
!pip install -q google-genai python-dotenv

import os
import datetime
import json
from getpass import getpass

# Check for API key. If not present in environment, prompt the user
if not os.environ.get("GEMINI_API_KEY"):
    print("GEMINI_API_KEY environment variable not detected.")
    os.environ["GEMINI_API_KEY"] = getpass("Please enter your Gemini API Key: ")
else:
    print("GEMINI_API_KEY successfully detected in environment.")

### Step 2: State and Memory Management (`state.py` equivalent)
This class manages the agent's memory. It stores tasks, the built study schedule, and the execution history/trace.

In [ ]:
class StudyPlannerState:
    def __init__(self, start_date: str = None, max_study_hours_per_day: float = 4.0):
        """
        State and memory management for the Study Planner Agent.
        """
        # Default start date to today's ISO date if not provided
        self.start_date = start_date or datetime.date.today().isoformat()
        self.max_study_hours_per_day = max_study_hours_per_day
        self.tasks = []      # List of dicts: {"name": str, "due": str, "hours_required": float}
        self.schedule = {}   # Dictionary of YYYY-MM-DD -> list of dicts: {"task": str, "hours": float}
        self.history = []    # Trace of agent steps: [{"step": int, "action": dict, "observation": str}]

    def add_task_to_state(self, name: str, due: str, hours_required: float = 3.0):
        """Helper method to append a task directly to the state."""
        self.tasks.append({
            "name": name,
            "due": due,
            "hours_required": hours_required
        })

    def to_dict(self) -> dict:
        """Serializes the state to a plain dictionary."""
        return {
            "start_date": self.start_date,
            "max_study_hours_per_day": self.max_study_hours_per_day,
            "tasks": self.tasks,
            "schedule": self.schedule,
            "history": self.history
        }

    def to_json(self) -> str:
        """Serializes the state to a JSON string."""
        return json.dumps(self.to_dict(), indent=2)

### Step 3: Pure Python Tools (`tools.py` equivalent)
These are the two tools available to the agent. They operate directly on the state:
1. `add_task(name, due, state)`: Validates the date, calculates estimated hours, and stores the task in memory.
2. `build_schedule(state)`: Implements Earliest Deadline First (EDF) scheduling, allocating daily study blocks before the task is due, respecting the max daily capacity, and returning warnings if deadlines are too tight.

In [ ]:
def add_task(name: str, due: str, state: StudyPlannerState) -> str:
    """
    Store a task with its due date in the planner state.
    """
    # Validate the date format
    try:
        due_date = datetime.date.fromisoformat(due)
    except ValueError:
        return f"Error: Due date '{due}' is not in YYYY-MM-DD format. Task was NOT added."
    
    # Check for duplicate tasks
    for existing in state.tasks:
        if existing["name"].lower() == name.lower() and existing["due"] == due:
            return f"Task '{name}' due on {due} already exists in the system."
            
    # Determine default hours based on keywords in task name
    name_lower = name.lower()
    if "exam" in name_lower or "test" in name_lower or "final" in name_lower:
        hours = 6.0
    elif "project" in name_lower or "presentation" in name_lower:
        hours = 4.0
    elif "quiz" in name_lower or "homework" in name_lower or "assignment" in name_lower:
        hours = 2.0
    else:
        hours = 3.0  # standard default

    state.add_task_to_state(name, due, hours_required=hours)
    return f"Success: Task '{name}' added with {hours} estimated study hours required, due on {due}."


def build_schedule(state: StudyPlannerState) -> dict:
    """
    Build a study schedule from all stored tasks.
    """
    if not state.tasks:
        state.schedule = {}
        return {
            "status": "success",
            "message": "No tasks are currently stored. Add tasks before building a schedule.",
            "schedule": {},
            "warnings": []
        }
        
    try:
        start_date_parsed = datetime.date.fromisoformat(state.start_date)
    except ValueError:
        return {
            "status": "error",
            "message": f"Error: Start date '{state.start_date}' in state is invalid YYYY-MM-DD.",
            "schedule": {},
            "warnings": []
        }
        
    # Order tasks by due date (Earliest Deadline First - EDF)
    try:
        sorted_tasks = sorted(state.tasks, key=lambda t: datetime.date.fromisoformat(t["due"]))
    except ValueError as e:
        return {
            "status": "error",
            "message": f"Error: One or more tasks have invalid due dates. Details: {str(e)}",
            "schedule": {},
            "warnings": []
        }
        
    schedule = {}         # YYYY-MM-DD -> list of dicts: {"task": name, "hours": float}
    daily_allocated = {}  # YYYY-MM-DD -> float (running total of hours allocated on that day)
    warnings = []
    
    for task in sorted_tasks:
        task_name = task["name"]
        due_date_parsed = datetime.date.fromisoformat(task["due"])
        hours_needed = task["hours_required"]
        
        # Calculate valid study days: starting from state.start_date up to the day BEFORE due_date
        study_days = []
        curr = start_date_parsed
        while curr < due_date_parsed:
            study_days.append(curr.isoformat())
            curr += datetime.timedelta(days=1)
            
        if not study_days:
            if due_date_parsed == start_date_parsed:
                study_days = [state.start_date]
            else:
                warnings.append(
                    f"Task '{task_name}' is due on {task['due']}, which is before the current schedule "
                    f"start date ({state.start_date}). No study blocks could be scheduled for it."
                )
                continue
                
        # Allocate study blocks (max of 2.0 hours per task per day for variety)
        allocated_hours = 0.0
        
        # Phase 1: Distribute in blocks of up to 2 hours
        for day in study_days:
            if allocated_hours >= hours_needed:
                break
            current_day_total = daily_allocated.get(day, 0.0)
            available_capacity = state.max_study_hours_per_day - current_day_total
            if available_capacity <= 0:
                continue
                
            to_allocate = min(hours_needed - allocated_hours, available_capacity, 2.0)
            if to_allocate > 0:
                if day not in schedule:
                    schedule[day] = []
                schedule[day].append({"task": task_name, "hours": to_allocate})
                daily_allocated[day] = current_day_total + to_allocate
                allocated_hours += to_allocate
                
        # Phase 2: Fill remaining capacity without the 2.0 hour limit if needed
        if allocated_hours < hours_needed:
            for day in study_days:
                if allocated_hours >= hours_needed:
                    break
                current_day_total = daily_allocated.get(day, 0.0)
                available_capacity = state.max_study_hours_per_day - current_day_total
                if available_capacity <= 0:
                    continue
                    
                to_allocate = min(hours_needed - allocated_hours, available_capacity)
                if to_allocate > 0:
                    if day not in schedule:
                        schedule[day] = []
                    existing = next((item for item in schedule[day] if item["task"] == task_name), None)
                    if existing:
                        existing["hours"] += to_allocate
                    else:
                        schedule[day].append({"task": task_name, "hours": to_allocate})
                    daily_allocated[day] = current_day_total + to_allocate
                    allocated_hours += to_allocate
                    
        if allocated_hours < hours_needed:
            warnings.append(
                f"Warning: Could only schedule {allocated_hours:.1f} of {hours_needed:.1f} hours for '{task_name}' "
                f"due to study limits (Max {state.max_study_hours_per_day}h/day)."
            )
            
    sorted_keys = sorted(schedule.keys())
    final_schedule = {day: schedule[day] for day in sorted_keys}
    
    state.schedule = final_schedule
    
    return {
        "status": "success" if not warnings else "warning",
        "schedule": final_schedule,
        "warnings": warnings,
        "message": f"Successfully built schedule with {len(warnings)} warnings."
    }

### Step 4: Custom Agent plan-act Loop (`agent.py` equivalent)
The function `run_agent` formats the state and execution history (memory) into a prompt, prompts Gemini in JSON mode, parses the JSON response, calls tools, updates state, and continues the loop until the final answer is obtained.

In [ ]:
from google import genai
from google.genai import types

TOOL_REGISTRY = {
    "add_task": add_task,
    "build_schedule": build_schedule
}

def get_system_instruction() -> str:
    return """You are a Study Planner Agent (Topic T24). Your goal is to plan study blocks around real deadlines.
You must run a plan-act loop, using tools to store tasks and build a schedule, rather than just answering directly.

You have access to the following tools:
1. `add_task(name: str, due: str)`
   - Stores a study task with its due date (YYYY-MM-DD format).
2. `build_schedule()`
   - Builds a chronological study schedule from all stored tasks.
   - Fits study blocks before each due date.

At each step, you must output a single JSON object. You have three possible actions:

1. Call a tool:
   {
     "thought": "Brief explanation of why you are calling this tool",
     "action": "call_tool",
     "tool_name": "add_task",
     "tool_args": {"name": "Task Name", "due": "YYYY-MM-DD"}
   }
   OR
   {
     "thought": "Brief explanation",
     "action": "call_tool",
     "tool_name": "build_schedule",
     "tool_args": {}
   }

2. Ask the user for clarification:
   {
     "thought": "Brief explanation",
     "action": "ask_user",
     "message": "Clarification question"
   }

3. Provide the final study plan / answer:
   {
     "thought": "Brief explanation",
     "action": "final_answer",
     "message": "A detailed, friendly summary of the final schedule"
   }

Strict Rules:
- You must output VALID JSON. No extra text or formatting blocks.
- You must call `add_task` for each task first, then call `build_schedule` before presenting `final_answer`.
"""

def format_agent_prompt(goal: str, state: StudyPlannerState) -> str:
    state_dict = state.to_dict()
    prompt = f"""### Current Start Date: {state.start_date}
### Daily Study Limit: {state.max_study_hours_per_day} hours/day

### User Goal:
\"{goal}\"

### Currently Stored Tasks:
{json.dumps(state_dict['tasks'], indent=2)}

### Current Execution Trace (Memory):
"""
    if not state.history:
        prompt += "No steps taken yet.\n"
    else:
        for step in state.history:
            prompt += f"Step {step['step']}:\n"
            prompt += f"  - Action Taken: {json.dumps(step['action'])}\n"
            prompt += f"  - Observation: {step['observation']}\n\n"
            
    prompt += "\nRespond with your next action in the required JSON format."
    return prompt

def run_agent(goal: str, state: StudyPlannerState, client, model: str = "gemini-2.5-flash", max_steps: int = 10) -> tuple:
    step_count = len(state.history) + 1
    
    for step_idx in range(max_steps):
        # 1. Format prompt showing goals, tasks, and memory/history
        prompt = format_agent_prompt(goal, state)
        
        try:
            # 2. Call the LLM in JSON mode
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=get_system_instruction(),
                    response_mime_type="application/json",
                    temperature=0.0
                )
            )
            
            # 3. Parse JSON response
            response_text = response.text.strip()
            response_json = json.loads(response_text)
            
            thought = response_json.get("thought", "")
            action = response_json.get("action", "")
            
            # 4. Decides next action (Viva Checkpoint)
            if action == "call_tool":
                tool_name = response_json.get("tool_name")
                tool_args = response_json.get("tool_args", {})
                
                if tool_name not in TOOL_REGISTRY:
                    observation = f"Error: Tool '{tool_name}' is not supported."
                else:
                    # Execute tool
                    tool_func = TOOL_REGISTRY[tool_name]
                    observation_result = tool_func(**tool_args, state=state)
                    observation = json.dumps(observation_result) if isinstance(observation_result, dict) else str(observation_result)
                
                state.history.append({
                    "step": step_count,
                    "thought": thought,
                    "action": {"action": action, "tool_name": tool_name, "tool_args": tool_args},
                    "observation": observation
                })
                print(f"[Step {step_count}] Called tool '{tool_name}' with args {tool_args}")
                print(f"         Observation: {observation}\n")
                
            elif action == "ask_user":
                message = response_json.get("message", "")
                state.history.append({
                    "step": step_count,
                    "thought": thought,
                    "action": {"action": action, "message": message},
                    "observation": "Waiting for user input."
                })
                print(f"[Step {step_count}] Agent asks user: {message}\n")
                return message, state.history
                
            elif action == "final_answer":
                message = response_json.get("message", "")
                state.history.append({
                    "step": step_count,
                    "thought": thought,
                    "action": {"action": action, "message": message},
                    "observation": "Plan successfully generated."
                })
                print(f"[Step {step_count}] Final Answer:\n{message}\n")
                return message, state.history
                
            else:
                raise ValueError(f"Unknown action: '{action}'")
                
        except json.JSONDecodeError:
            err_msg = "Error: Failed to parse your response as JSON. Make sure you return pure JSON."
            state.history.append({
                "step": step_count,
                "thought": "JSON parsing failed.",
                "action": {"action": "parse_failure"},
                "observation": err_msg
            })
            print(f"[Step {step_count}] JSON parsing failed. Retrying...\n")
            
        except Exception as e:
            err_msg = f"Error: An exception occurred: {str(e)}"
            state.history.append({
                "step": step_count,
                "thought": "Internal error.",
                "action": {"action": "internal_error"},
                "observation": err_msg
            })
            print(f"[Step {step_count}] System error: {str(e)}\n")
            
        step_count += 1
        
    return "Error: Agent reached maximum steps.", state.history

### Step 5: Simulation Client Helper
This cell defines the `MockGenAIClient` used for testing without an API key or internet access.

In [ ]:
class MockResponse:
    def __init__(self, text):
        self.text = text

class MockModels:
    def __init__(self, responses):
        self.responses = responses
        self.call_count = 0
        
    def generate_content(self, model, contents, config=None):
        if self.call_count < len(self.responses):
            res = MockResponse(self.responses[self.call_count])
            self.call_count += 1
            return res
        else:
            return MockResponse('{"thought": "Default complete", "action": "final_answer", "message": "Planning complete."}')

class MockGenAIClient:
    def __init__(self, responses):
        self.models = MockModels(responses)

### Step 6: Demo runs (Runs 2 to 3 Goals)

#### Goal 1: Standard Schedule Planning
Let's ask the agent to plan study blocks for a **Math Exam** due on 2026-08-14, **History Homework** due on 2026-08-13, and **Physics Quiz** due on 2026-08-12.

In [ ]:
MOCK_RESPONSES_GOAL_1 = [
    '{"thought": "I see three tasks in the goal. I should add the Physics Quiz first since it is due earliest on 2026-08-12.", "action": "call_tool", "tool_name": "add_task", "tool_args": {"name": "Physics Quiz", "due": "2026-08-12"}}',
    '{"thought": "Now I will add the History Homework, which is due on 2026-08-13.", "action": "call_tool", "tool_name": "add_task", "tool_args": {"name": "History Homework", "due": "2026-08-13"}}',
    '{"thought": "Next, I need to add the Math Exam, due on 2026-08-14.", "action": "call_tool", "tool_name": "add_task", "tool_args": {"name": "Math Exam", "due": "2026-08-14"}}',
    '{"thought": "All tasks are now stored. I will run build_schedule to calculate the optimal daily study blocks.", "action": "call_tool", "tool_name": "build_schedule", "tool_args": {}}',
    '{"thought": "The schedule has been successfully built. I will now present the finalized study plan to the user.", "action": "final_answer", "message": "Here is your study plan starting 2026-08-11:\\n\\n- **2026-08-11 (Day 1)**:\\n  * Physics Quiz (2.0 hours) - review before tomorrow\'s quiz\\n  * History Homework (2.0 hours) - write homework due 2026-08-13\\n  *Total: 4.0 hours*\\n\\n- **2026-08-12 (Day 2)**:\\n  * Math Exam (4.0 hours) - study block 1 for exam on 2026-08-14\\n  *Total: 4.0 hours*\\n\\n- **2026-08-13 (Day 3)**:\\n  * Math Exam (2.0 hours) - study block 2 / final review\\n  *Total: 2.0 hours*\\n\\nAll study blocks have been allocated before the deadlines, respecting your daily limit of 4 hours."}'
]

# Initialize State for Goal 1
state1 = StudyPlannerState(start_date="2026-08-11")
goal1 = "I need to prepare for my upcoming assessments: a Math Exam due on 2026-08-14, History Homework due on 2026-08-13, and Physics Quiz due on 2026-08-12."

# Determine client
api_key = os.environ.get("GEMINI_API_KEY")
if api_key:
    print("--- RUNNING LIVE WITH GEMINI API ---")
    client = genai.Client(api_key=api_key)
else:
    print("--- RUNNING IN SIMULATION MODE (No API Key) ---")
    client = MockGenAIClient(MOCK_RESPONSES_GOAL_1)

run_agent(goal1, state1, client)

#### Goal 2: Re-planning with Memory
To demonstrate memory, we will **reuse the exact state from Goal 1** (which contains our existing tasks). We tell the agent: *"Wait, I just found out I also have a Chemistry Lab due tomorrow (2026-08-12). Re-plan!"*

The agent reads the existing tasks from memory, adds the new task, rebuilds the schedule, and returns a new plan that fits the new deadline.

In [ ]:
MOCK_RESPONSES_GOAL_2 = [
    '{"thought": "I need to add the new Chemistry Lab task which is due tomorrow on 2026-08-12.", "action": "call_tool", "tool_name": "add_task", "tool_args": {"name": "Chemistry Lab", "due": "2026-08-12"}}',
    '{"thought": "Now that the Chemistry Lab is added alongside the existing Math, History, and Physics tasks, I must rebuild the schedule.", "action": "call_tool", "tool_name": "build_schedule", "tool_args": {}}',
    '{"thought": "The schedule rebuild was successful. Let\'s report the new plan and note that because of the new urgent deadline, we have a capacity warnings on Day 1.", "action": "final_answer", "message": "Re-planning complete! Here is your updated study plan:\\n\\n- **2026-08-11 (Day 1)**:\\n  * Physics Quiz (2.0 hours) - due tomorrow\\n  * Chemistry Lab (2.0 hours) - due tomorrow\\n  *(Note: Day 1 capacity of 4 hours is fully hit; History Homework has been shifted to Day 2)*\\n\\n- **2026-08-12 (Day 2)**:\\n  * History Homework (2.0 hours) - due 2026-08-13\\n  * Math Exam (2.0 hours) - study block 1\\n\\n- **2026-08-13 (Day 3)**:\\n  * Math Exam (4.0 hours) - study block 2\\n\\n**Warnings / Capacity Conflicts:**\\n- Warning: Could only schedule 4.0 of 6.0 hours for \'Math Exam\' because Day 1 and Day 2 capacities were filled with closer deadlines. Please consider extending study hours or asking for a Math extension."}'
]

goal2 = "Wait, I just found out I also have a Chemistry Lab report due tomorrow (2026-08-12). Re-plan my schedule!"

if api_key:
    print("--- RUNNING LIVE WITH GEMINI API (REUSING STATE) ---")
    client = genai.Client(api_key=api_key)
else:
    print("--- RUNNING IN SIMULATION MODE (No API Key; REUSING STATE) ---")
    client = MockGenAIClient(MOCK_RESPONSES_GOAL_2)

run_agent(goal2, state1, client)

### Step 7: Print Complete Memory / History Trace
Below we print the entire step-by-step trace stored in the state history. This is the **proof** that the agent is taking multiple steps, checking tool results, and maintaining session memory across multiple turns.

In [ ]:
print("=" * 60)
print("     COMPLETE STUDY PLANNER AGENT CONVERSATION HISTORY")
print("=" * 60)
for step in state1.history:
    print(f"Step {step['step']}:")
    print(f"  Thought: {step['thought']}")
    print(f"  Action: {step['action']}")
    print(f"  Observation: {step['observation']}")
    print("-" * 60)